# Regression Discontinuity Design (RDD)

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [ ]:
"""Shared setup for all chapters: imports, plotting theme, and data generators."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

rng = np.random.default_rng(42)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")


def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # which markets get the program
    treat_week=60,              # program launch week
    true_effect=8.0,            # true lift in weekly revenue ($k)
    effect_ramp=6,              # weeks for the effect to reach full size
    seed=42,
):
    """NorthShore Retail store-week panel with known treatment effect."""
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets
    store_fe = rng.normal(100, 15, n_stores)
    market_fe = rng.normal(0, 8, n_markets)
    store_trend = rng.normal(0.05, 0.03, n_stores)

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks))
    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets
            post = w >= treat_week
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])


def make_customers(n=8000, true_att=12.0, seed=42):
    """Loyalty-program adoption data with selection on observables."""
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)
    spend0 = rng.lognormal(4.0, 0.5, n)
    engage = rng.beta(2, 5, n) * 10
    urban = rng.binomial(1, 0.55, n)
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))


panel = make_panel()
cust = make_customers()
X_cols = ["tenure", "spend0", "engage", "urban"]
TRUE_ATT = 12.0


**Business setting:** treatment is assigned by a **threshold on a continuous score**. These are everywhere in business:

- Customers with churn-risk score ≥ 0.7 get a retention call.
- Orders over $50 get free shipping.
- Stores below a performance score get the turnaround program.
- Loan applicants above a credit score get approved.

**Logic:** units *just below* and *just above* the cutoff are essentially identical except for treatment — assignment near the threshold is as good as random. Comparing outcomes in a narrow window around the cutoff identifies the **local** effect *at the threshold*.

**Identifying assumptions:** (1) potential outcomes are continuous in the running variable at the cutoff (nothing else jumps there), and (2) units cannot precisely manipulate their score to sort around the cutoff.

## Simulated example: retention calls triggered by churn score

In [ ]:
n = 6000
score = rng.uniform(0, 1, n)                     # churn-risk score
D = (score >= 0.7).astype(int)                   # call if score >= 0.7
TRUE_RD = 6.5                                    # call adds 6.5pp retention value ($)
y = 40 - 25*score + 8*score**2 + TRUE_RD*D + rng.normal(0, 5, n)
rd = pd.DataFrame(dict(score=score, D=D, y=y))

## Visualize with binned means

Always plot first — a discontinuity you can't see in a binned scatter is rarely robust:

In [ ]:
rd["bin"] = pd.cut(rd.score, bins=40)
binned = rd.groupby("bin", observed=True).agg(
    x=("score", "mean"), y=("y", "mean"))
plt.figure(figsize=(9, 4))
plt.scatter(binned.x, binned.y, s=18)
plt.axvline(0.7, color="red", ls="--")
plt.xlabel("Churn-risk score"); plt.ylabel("Customer value ($)")
plt.tight_layout(); plt.show()

## Robust local-polynomial estimation with `rdrobust`

State of the art (Calonico, Cattaneo & Titiunik) is local linear regression within a **data-driven bandwidth**, with bias-corrected robust confidence intervals:

In [ ]:
%pip install rdrobust
from rdrobust import rdrobust, rdplot

out = rdrobust(y=rd.y, x=rd.score, c=0.7)
print(out)                       # conventional + robust bias-corrected CIs
rdplot(y=rd.y, x=rd.score, c=0.7)

## Validity checks

- **Manipulation test (McCrary / density test):** if salespeople nudge customers' scores over the line, the score's density jumps at the cutoff. Test with `rddensity`:

In [ ]:
%pip install rddensity
from rddensity import rddensity
print(rddensity(rd.score, c=0.7))

- **Covariate continuity:** pre-treatment covariates (tenure, baseline spend) should *not* jump at the cutoff — run `rdrobust` with each covariate as the outcome.
- **Placebo cutoffs:** estimate at fake thresholds (0.5, 0.6, 0.8); effects there should be ≈ 0.
- **Bandwidth sensitivity:** report estimates across a range of bandwidths.

## Fuzzy RDD

If crossing the threshold only *increases the probability* of treatment (agents don't always make the call; some below-threshold customers get called anyway), the design is **fuzzy**: the threshold becomes an instrument for treatment, and the estimand is a LATE at the cutoff. `rdrobust(y, x, c=0.7, fuzzy=rd.D)` handles this directly.

**Interpretation caveat for business audiences:** RDD identifies the effect *for units near the cutoff*. "The retention call is worth $6.5 for customers around score 0.7" does not license "call everyone." It does, however, answer the exactly-right marginal question: *should we move the threshold?*